In [0]:
AZURE_STORAGE_ACCOUNT_NAME=""
AZURE_STORAGE_ACCOUNT_KEY=""
AZURE_CONTAINER_NAME=""

spark.conf.set(f"fs.azure.account.key.{AZURE_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", AZURE_STORAGE_ACCOUNT_KEY)

raw_path = f"abfss://{AZURE_CONTAINER_NAME}@{AZURE_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/raw/real_estate/*/*/*/*.json"

df_raw = spark.read.option("multiLine", True).json(raw_path)

df_raw.display()
df_raw.printSchema()

areaType,bathrooms,bedrooms,description,floorSize,id,pageUrl,possessionStatus,postedBy,pricePerSqft,priceRange,propertyType,scrapedAt,title,url
1 Bath,65 sqm,700 sqft,"This lovely 1 bhk apartment for rent in powai is available in one of central mumbai's most popular godrej urbna park powai . This 1 bhk flat comes with 1 bathroom set in 650 sq.Ft. Super built-Up usable area. The flat consists of 1 bedroom, built on 700 sq.Ft. A separate space for store room is available in this flat. Society provides 1 covered parking with this flat. The building has a total o...",1 BHK,H83562922,https://www.99acres.com/search/property/rent/serviced-apartments/central-mumbai?city=14&property_type=22&preference=R&area_unit=1&res_com=R&isPreLeased=N,,Sai Realtors,"/month+ Deposit ₹1,50,000","₹52,000 /month",1 BHK Serviced Apartment for rent,2026-03-08T12:44:23.028Z,"1 BHK Serviced Apartment for rent in Chandivali, Mumbai",https://www.99acres.com/1-bhk-bedroom-serviced-apartment-flat-for-rent-in-godrej-urban-park-chandivali-central-mumbai-suburbs-700-sq-ft-r3-spid-H83562922
3 Baths,123 sqm,"1,320 sqft","Located inside the bps compound hence quiet and peace jain temple situated in front of the property Other facilities like garden ,market , station etc near by",3 BHK,S89384529,https://www.99acres.com/search/property/rent/serviced-apartments/central-mumbai?city=14&property_type=22&preference=R&area_unit=1&res_com=R&isPreLeased=N,,Owner,/month,"₹68,000 /month",3 BHK Flat for rent,2026-03-08T12:44:23.040Z,"3 BHK Flat for rent in Mulund West, Mumbai",https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-rent-in-blue-bells-chs-mulund-west-central-mumbai-suburbs-1320-sq-ft-spid-S89384529
2 Baths,70 sqm,750 sqft,"Looking for a premium lifestyle in powai? This 12th-Floor lake-Facing 2 bhk offers peace, ventilation, and unbeatable connectivity perfect for working professionals & families. Prime location powai 750 sq. Ft. Carpet area High floor | bright | breezy | private What makes this home special? Stunning open lake view 2 bedrooms with air conditioners 2 geysers installed Stylish modular kitchen Light...",2 BHK,K89012398,https://www.99acres.com/search/property/rent/serviced-apartments/central-mumbai?city=14&property_type=22&preference=R&area_unit=1&res_com=R&isPreLeased=N,,Owner,/month,"₹79,999 /month",2 BHK Flat for rent,2026-03-08T12:44:23.041Z,"2 BHK Flat for rent in Powai, Mumbai",https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-rent-in-powai-central-mumbai-suburbs-750-sq-ft-spid-K89012398
1 Bath,33 sqm,350 sqft,"It's pg (Paying guest) for working girls. It's on bed basis, i have double sharing and triple sharing bed.Now bed is available in triple sharing which is in hall. Amenities like water purifier, geyser, washing machine, fridge,single bed with mattress and pillow,single godrej cupboard is there. Kitchen with gas is provided to cook .Single toilet n single bathroom is there for 5 girls. Ghatkopar ...",1 BHK,H89278691,https://www.99acres.com/search/property/rent/serviced-apartments/central-mumbai?city=14&property_type=22&preference=R&area_unit=1&res_com=R&isPreLeased=N,,Owner,/month+ Deposit 1 months rent,"₹9,000 /month",1 BHK Flat for rent,2026-03-08T12:44:23.042Z,"1 BHK Flat for rent in Ghatkopar West, Mumbai",https://www.99acres.com/1-bhk-bedroom-apartment-flat-for-rent-in-damodar-park-ghatkopar-west-central-mumbai-suburbs-350-sq-ft-spid-H89278691
1 Bath,23 sqm,250 sqft,"This unfurnished 1 bhk flat in navjeevan society, chembur east, offers 250 sq.Ft. On the top floor of a low-Rise, 1-5 year old building. With 1 bedroom and 1 bathroom, its available for immediate rent. Enjoy proximity to several religious landmarks, blending convenience with a peaceful neighborhood setting.",1 BHK,P88822596,https://www.99acres.com/search/property/rent/serviced-apartments/central-mumbai?city=14&property_type=22&preference=R&area_unit=1&res_com=R&isPreLeased=N,,Owner,/month,"₹10,000 /month",1 BHK Flat for rent,2026-03-08T12:44:23.043Z,"1 BHK Flat fo

root
 |-- areaType: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- description: string (nullable = true)
 |-- floorSize: string (nullable = true)
 |-- id: string (nullable = true)
 |-- pageUrl: string (nullable = true)
 |-- possessionStatus: string (nullable = true)
 |-- postedBy: string (nullable = true)
 |-- pricePerSqft: string (nullable = true)
 |-- priceRange: string (nullable = true)
 |-- propertyType: string (nullable = true)
 |-- scrapedAt: string (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)



In [0]:
from pyspark.sql.functions import col

df = df_raw.select(
    col("id").alias("property_id"),
    col("title"),
    col("propertyType"),
    col("priceRange"),
    col("bedrooms"),
    col("bathrooms"),
    col("postedBy"),
    col("scrapedAt")
)

In [0]:
from pyspark.sql.functions import regexp_replace

df = df.withColumn(
    "price",
    regexp_replace("priceRange", "[^0-9]", "").cast("int")
)

In [0]:
from pyspark.sql.functions import regexp_extract

df = df.withColumn(
    "bhk",
    regexp_extract("propertyType", r"(\d+)", 1).cast("int")
)

In [0]:
df = df.withColumn(
    "area_sqft",
    regexp_replace("bedrooms", "[^0-9]", "").cast("int")
)

In [0]:
df = df.withColumn(
    "bathrooms",
    regexp_replace("bathrooms", "[^0-9]", "").cast("int")
)

In [0]:
df.display()

property_id,title,propertyType,priceRange,bedrooms,bathrooms,postedBy,scrapedAt,price,bhk,area_sqft
H83562922,"1 BHK Serviced Apartment for rent in Chandivali, Mumbai",1 BHK Serviced Apartment for rent,"₹52,000 /month",700 sqft,65,Sai Realtors,2026-03-08T12:44:23.028Z,52000,1,700
S89384529,"3 BHK Flat for rent in Mulund West, Mumbai",3 BHK Flat for rent,"₹68,000 /month","1,320 sqft",123,Owner,2026-03-08T12:44:23.040Z,68000,3,1320
K89012398,"2 BHK Flat for rent in Powai, Mumbai",2 BHK Flat for rent,"₹79,999 /month",750 sqft,70,Owner,2026-03-08T12:44:23.041Z,79999,2,750
H89278691,"1 BHK Flat for rent in Ghatkopar West, Mumbai",1 BHK Flat for rent,"₹9,000 /month",350 sqft,33,Owner,2026-03-08T12:44:23.042Z,9000,1,350
P88822596,"1 BHK Flat for rent in Chembur East, Mumbai",1 BHK Flat for rent,"₹10,000 /month",250 sqft,23,Owner,2026-03-08T12:44:23.043Z,10000,1,250
E89248095,"1 BHK Flat for rent in Chembur, Mumbai",1 BHK Flat for rent,"₹7,000 /month",250 sqft,23,Owner,2026-03-08T12:44:23.043Z,7000,1,250
E89053094,"1 BHK Flat for rent in Bhandup East, Mumbai",1 BHK Flat for rent,"₹8,000 /month",200 sqft,19,Owner,2026-03-08T12:44:23.044Z,8000,1,200
Q89384247,"2 BHK Flat for rent in Powai, Mumbai",2 BHK Flat for rent,"₹80,000 /month",690 sqft,64,Owner,2026-03-08T12:44:23.045Z,80000,2,690
X89335577,"2 BHK Flat for rent in Ghatkopar West, Mumbai",2 BHK Flat for rent,"₹66,000 /month",650 sqft,60,Owner,2026-03-08T12:44:23.046Z,66000,2,650
X89016720,"3 BHK Flat for rent in Chembur, Mumbai",3 BHK Flat for rent,₹1.1 Lac /month,"1,200 sqft",111,Owner,2026-03-08T12:44:23.046Z,11,3,1200


In [0]:
silver_path = f"abfss://{AZURE_CONTAINER_NAME}@{AZURE_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/silver/properties"

df.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
from pyspark.sql.functions import current_date, lit

silver_df = spark.read.format("delta").load(
    f"abfss://{AZURE_CONTAINER_NAME}@{AZURE_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/silver/properties"
)

history_df = silver_df \
    .withColumn("valid_from", current_date()) \
    .withColumn("valid_to", lit(None)) \
    .withColumn("is_current", lit(True))

history_df.write.format("delta") \
    .mode("overwrite") \
    .save(f"abfss://{AZURE_CONTAINER_NAME}@{AZURE_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/property_price_history")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date

gold_path = f"abfss://{AZURE_CONTAINER_NAME}@{AZURE_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/property_price_history"

silver_df = spark.read.format("delta").load(
    f"abfss://{AZURE_CONTAINER_NAME}@{AZURE_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/silver/properties"
)

# Fix valid_to column (written as void/NullType by upstream cell)
gold_schema = spark.read.format("delta").load(gold_path).schema
if "valid_to" not in [f.name for f in gold_schema.fields]:
    try:
        spark.sql(f"ALTER TABLE delta.`{gold_path}` SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')")
        spark.sql(f"ALTER TABLE delta.`{gold_path}` DROP COLUMN valid_to")
    except Exception:
        pass
    spark.sql(f"ALTER TABLE delta.`{gold_path}` ADD COLUMNS (valid_to DATE)")

history_table = DeltaTable.forPath(
    spark,
    gold_path
)

(
history_table.alias("t")
.merge(
    silver_df.alias("s"),
    "t.property_id = s.property_id AND t.is_current = true"
)

.whenMatchedUpdate(
    condition="t.price != s.price",
    set={
        "valid_to": "current_date()",
        "is_current": "false"
    }
)

.whenNotMatchedInsert(
    values={
        "property_id": "s.property_id",
        "price": "s.price",
        "bhk": "s.bhk",
        "area_sqft": "s.area_sqft",
        "valid_from": "current_date()",
        "valid_to": "NULL",
        "is_current": "true"
    }
)

.execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]